In [ ]:
# Find the pipeline package without walking the DICOM tree.
import sys, os, time, gc, json
from pathlib import Path

T0 = time.time()
def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)

def walk(root="/kaggle/input", max_depth=8, suffixes=None):
    base = len(Path(root).parts)
    for dirpath, dirnames, filenames in os.walk(root):
        here = Path(dirpath)
        if len(here.parts) - base >= max_depth:
            dirnames[:] = []
        else:
            dirnames[:] = [d for d in dirnames if not d.lower().endswith(("_series", "_images"))]
        for f in filenames:
            if suffixes is None or Path(f).suffix.lower() in suffixes:
                yield here / f

PKG = next((p.parent.parent for p in walk(suffixes=(".py",))
            if p.name == "targets.py" and p.parent.name == "rsna_knee"), None)
if PKG is None:
    raise FileNotFoundError("attach the rsna-knee-pipeline dataset (v2, it must contain rsna_knee/targets.py)")
sys.path.insert(0, str(PKG))

import numpy as np
import pandas as pd
from rsna_knee.report_labels import TARGETS
log(f"pipeline: {PKG}")


In [ ]:
# CPU notebook (Accelerator: None). Run once per shard: SHARD = 0, 1, 2.
SHARD, N_SHARDS = 0, 3

from rsna_knee.infer import find_root
from rsna_knee.data import assign_folds
from rsna_knee.cache3 import build_split

ROOT = find_root()
train = pd.read_csv(ROOT / "train.csv")
train["fold"] = assign_folds(train, n_folds=5, seed=2026)
OUT = Path("/kaggle/working")
train.to_csv(OUT / "train_folds.csv", index=False)
build_split(ROOT, "train", train.StudyInstanceUID.tolist(), OUT, shard=SHARD, n_shards=N_SHARDS, log=log)
log("done. Save the output as a Dataset: rsna-knee-cache-v3-" + str(SHARD))
